In [18]:
import GalerkinToolkit as GT
import PartitionedArrays as PA
import MPI; MPI.Init()
import JSON
using LinearAlgebra

### Step 1: Define the ranks
Set up a DebugArray with an appropriate number of ranks. 

In [3]:
np = 3
ranks = PA.DebugArray(1:np)

3-element PartitionedArrays.DebugArray{Int64, 1}:
[1] = 1
[2] = 2
[3] = 3


### Step 2: Define the setup function 
Setup function is used to prepare everything needed for RHS computation. This includes: mesh, faces, nodes, nodes coordinates etc. \
Notation used: 
1. `gn`	global node
2. `ln`	local node
3. `gf`	global face/element
4. `lf`	local face/element
5. `fn`	face node
6. `fq`	face quadrature point
7. `gq`	global quadrature point

In [4]:
function do_setup(nx,ny)
    mesh = GT.cartesian_mesh((0,1,0,1),(nx,ny))
    Ω = GT.interior(mesh)
    dΩ = GT.quadrature(Ω,2)
    V = GT.lagrange_space(Ω,2)
    V_faces= GT.each_face(V,dΩ;tabulate=(GT.value,GT.gradient))
    fq_fn_I = transpose(V_faces.accessor.reference_space_face.workspace.values[1])
    fq_fn_dI = transpose(V_faces.accessor.reference_space_face.workspace.gradients[1])
    fq_refdy = GT.weights(GT.reference_quadratures(dΩ)[1])
    gf_fn_gn = GT.face_nodes(V)
    gn_x = GT.node_coordinates(V)
    nfq = size(fq_fn_I,1)
    ngf = length(gf_fn_gn)
    ngq = nfq*ngf
    ngn = length(gn_x)
    (;fq_fn_I,fq_fn_dI,fq_refdy,gf_fn_gn,gn_x)
end    

do_setup (generic function with 1 method)

### Step 3: Define restrict setup
This function creates a data needed for each rank. The goal is for each (pretend) rank to have any the data necessay for it. 

In [ ]:
function restrict_setup(setup, ln_gn, gn_ln)
    (;fq_fn_I, fq_fn_dI, fq_refdy, gf_fn_gn, gn_x) = setup

    ln_gn = collect(ln_gn)

    ln_x = gn_x[ln_gn]

    ngf = length(gf_fn_gn)
    ngn = length(gn_x)

    gn_in_part = fill(false, ngn)
    gn_in_part[ln_gn] .= true

    gf_in_part = fill(false, ngf)

    for gf in 1:ngf
        fn_gn = gf_fn_gn[gf]

        for gn in fn_gn
            if gn_in_part[gn]
                gf_in_part[gf] = true
                break
            end
        end
    end

    lf_gf = findall(gf_in_part)

    lf_fn_gn = PA.jagged_array(gf_fn_gn[lf_gf])

    data = copy(lf_fn_gn.data)

    for k in 1:length(data)
        gn = data[k]
        data[k] = gn_ln[gn]
    end

    ptrs = lf_fn_gn.ptrs
    lf_fn_ln = PA.jagged_array(data, ptrs)

    nfq, nfn = size(fq_fn_I)
    ngq = ngf * nfq
    nln = length(ln_x)

    # Interpolated z_L values at global quadrature points
    gq_zl = zeros(ngq, nln)

    # Full global u vector, needed later for dz_1.
    gn_u = zeros(ngn)
    fn_ln_zl = zeros(nfn, nln)
    fq_ln_zl_tilde = zeros(nfq, nln)

    return (;
        fq_fn_I,
        fq_fn_dI,
        fq_refdy,
        gf_fn_gn,
        gn_x,
        ln_gn,
        ln_x,
        ngf,
        ngn,
        lf_gf,
        lf_fn_gn,
        lf_fn_ln,
        gq_zl,
        gn_u,
        fn_ln_zl,
        fq_ln_zl_tilde,
    )
end

### Step 4: Define initial conditions 
Define initial conditions for both u and all z. 

In [ ]:
function setup_ln_u0_z0(setup, num_layer)
    (;ln_x, gn_x, ngn) = setup # each process needs global node coordinates to initialize z
    ln_u0 = φ.(ln_x) # initialize u
    gn_0 = φ.(gn_x)
    z_0 = gn_0 * transpose(ln_u0) # initialize z layer
    ln_all_z_0 = [copy(z_0) for _ in 1:num_layer]
    return ln_u0, ln_all_z_0      
end

setup_ln_u0_z0 (generic function with 2 methods)

### Step 5: Setup on main 
In this step we create all rank specific setup on the main process. Then scatter the setup to other (pretend) rank. 

In [ ]:
function prepare_setups_on_main(np,nx,ny)
    setup = do_setup(nx,ny)
    (;gn_x) = setup
    ngn = length(gn_x) # number of global nodes 
    gn_partition_sequential = PA.uniform_partition(1:np,np,ngn)
    p_fem_setup = map(gn_partition_sequential) do nids
        ln_gn = PA.local_to_global(nids)
        gn_ln = PA.global_to_local(nids)
        restrict_setup(setup,ln_gn,gn_ln)
    end
end

prepare_setups_on_main (generic function with 1 method)

### Step 6: Define rhs functions

We define two rhs functions rhs_I! and rhs_W! (Why?)

In [ ]:
function rhs_I!(setup, gn_ln_zl)
    (;fq_fn_I, gf_fn_gn, gq_zl, fn_ln_zl, fq_ln_zl_tilde,ngf)= setup

    nfq, nfn = size(fq_fn_I)
    nln = size(gn_ln_zl, 2)

    fill!(gq_zl, 0)

    for gf in 1:ngf # loop over all faces
        fn_gn = gf_fn_gn[gf]

        for fn in 1:nfn
            gn = fn_gn[fn]
            for ln in 1:nln
                fn_ln_zl[fn,ln] = gn_ln_zl[gn, ln]
            end     
        end     
        mul!(fq_ln_zl_tilde, fq_fn_I, fn_ln_zl)
        
        # store z_L tilde at the global quadrature locations 
        for fq in 1:nfq
            gq = (gf-1)*nfq + fq # converts local quadrature indeks to global quatrature index
            for ln in 1:nln
                gq_zl[gq, ln] = fq_ln_zl_tilde[fq, ln]
            end     
        end
    end
end

# function rhs_W!(setup, ln_du)
#     (;fq_fn_I,fq_fn_dI,fq_refdy,lf_fn_ln,ln_x,ngf,ngn,gq_u,lf_gf,gf_fn_gn,gn_x) = setup
    
#     fill!(ln_du,0)

#     nln = length(ln_x)
#     nfq,nfn = size(fq_fn_I)
#     ngq = ngf * nfq # do we need this?
#     Ty = eltype(gn_x) # type of coordinates stored in gn_x 
#     zy = zero(Ty) # zero coordinate vectore
#     zJt = zero(zy*transpose(zy)) # zero matrix-like object for the Jacobian transpose 
#     fn_y = zeros(Ty,nfn) # physical coordinates of the nodes of one face 
#     fq_y = zeros(Ty,nfq) # physical coordinates of the quadrature points of one face
#     fq_dy = zeros(nfq) # physical quadrature weights for one face.
#     for gf in 1:ngf
#         fn_gn = gf_fn_gn[gf]
#         for fn in 1:nfn
#             gn = fn_gn[fn]
#             fn_y[fn] = gn_x[gn]
#         end
#         for fq in 1:nfq
#             y = zy
#             for fn in 1:nfn
#                 y += fq_fn_I[fq,fn]*fn_y[fn]
#             end
#             fq_y[fq] = y
#         end
#         for fq in 1:nfq
#             Jt = zJt
#             for fn in 1:nfn
#                 Jt += fn_y[fn]*fq_fn_dI[fq,fn]
#             end
#             refdy = fq_refdy[fq]
#             fq_dy[fq] = abs(det(Jt))*refdy
#         end
#         for fq in 1:nfq
#             y = fq_y[fq]
#             dy = fq_dy[fq]
#             gq = (gf-1)*nfq + fq
#             u = gq_u[gq] # this needs to be z
#             for ln in 1:nln
#                 x = ln_x[ln]
#                 W = norm(y-x)*dy # Use the right function here
#                 ln_du[ln] += W*u # Use firing rate here
#             end
#         end
#     end
# end

In [ ]:
function rhs_W!(setup, ln_du)
    (;fq_fn_I, fq_fn_dI, fq_refdy, gf_fn_gn, gn_x, ln_gn, ln_x, ngf, ngn, lf_gf, lf_fn_gn, lf_fn_ln, gq_zl, gn_u, fn_ln_zl, fq_ln_zl_tilde) = setup
    
    fill!(ln_du,0)

    nln = length(fq_ln_zl_tilde,2)
    nfq,nfn = size(fq_fn_I)
    Ty = eltype(gn_x) # type of coordinates stored in gn_x 
    zy = zero(Ty) # zero coordinate vectore
    zJt = zero(zy*transpose(zy)) # zero matrix-like object for the Jacobian transpose 
    fn_y = zeros(Ty,nfn) # physical coordinates of the nodes of one face 
    fq_y = zeros(Ty,nfq) # physical coordinates of the quadrature points of one face
    fq_dy = zeros(nfq) # physical quadrature weights for one face.
    #   ngq = ngf * nfq # do we need this?

    for ln in 1:nln
        for gf in 1:ngf
            fn_gn = gf_fn_gn[gf]
            for fn in 1:nfn
                gn = fn_gn[fn]
                fn_y[fn] = gn_x[gn]
            end
            for fq in 1:nfq
                y = zy
                for fn in 1:nfn
                    y += fq_fn_I[fq,fn]*fn_y[fn]
                end
                fq_y[fq] = y
            end
            for fq in 1:nfq
                Jt = zJt
                for fn in 1:nfn
                    Jt += fn_y[fn]*fq_fn_dI[fq,fn]
                end
                refdy = fq_refdy[fq]
                fq_dy[fq] = abs(det(Jt))*refdy
            end
            for fq in 1:nfq
                y = fq_y[fq]
                x = ln_x[ln]
                dy = fq_dy[fq]
                
                # calculate an adequate W entry for a given quadrature point and given local node
                wnq = w(x,y)*dy 
                zqn = fq_ln_zl_tilde[y,x]
                ln_du[ln] =+ wnq * f(zqn)
            end
        end
    end     
end

In [ ]:
# It is a good idea to measure the time for all lines
function rhs!(duz,uz,p_setup,elap_rhs)
    elap[1] = @elapsed foreach(setup->fill!(setup.gq_u,0),p_setup) # This is negligible as long as N/P >> 1
    elap[2] = @elapsed p_ln_u = PA.local_values(u)
    elap[3] = @elapsed foreach(rhs_I!,p_setup,p_ln_u)
    elap[4] = @elapsed p_gq_u = map(setup->setup.gq_u,p_setup)
    elap[5] = @elapsed all_reduce!(p_gq_u)
    elap[6] = @elapsed p_ln_du = PA.local_values(du)
    elap[7] = @elapsed foreach(rhs_W!,p_setup,p_ln_du)
end

### Step 6: Main function to run and time rhs 

In [ ]:
function main(backend,np,nx,ny,title)
    ranks = backend(1:np)

    # Setup
    elap_setup = zeros(2)
    elap_setup[1] = @elapsed p_setup_on_main = PA.map_main(ranks) do _ # setup data on the main process  
                                 prepare_setups_on_main(np,nx,ny)
                             end
    elap_setup[2] = @elapsed p_setup = PA.scatter(p_setup_on_main) # distribute data other processess 

    mem = Base.summarysize(p_setup) # calulate memory usage 

    # Initial conditions
    num_layer = 2
    p_ln_u0_all_z0 = map(setup -> setup_ln_u0_z0(setup, num_layer), p_setup)
    
    p_ln_u0 = map(data -> data[1], p_ln_u0_all_z0) # u
    p_all_z0 = map(data -> data[2], p_ln_u0_all_z0) # z

    ngn = PA.getany(map(setup->setup.ngn,p_setup))
    gn_partition = PA.uniform_partition(ranks,np,ngn) # partition of global node indices 

    u0 = PA.PVector(p_ln_u0, gn_partition)
    p_z0_layers = [
        map(all_z0 -> all_z0[layer], p_all_z0)
        for layer in 1:num_layer
    ]

    uz0 = ArrayPartition(u0, p_z0_layers...)

    uz = similar(uz0)
    duz = similar(uz0)

    nr = 10
    r_elap_rhs = [zeros(7) for _ in 1:nr]

    for r in 1:nr
        elap_rhs = r_elap_rhs[r]
        copy!(uz,uz0) # repeatedly called on the same data 
        rhs!(duz,uz,p_setup,elap_rhs)
    end

    elap = Dict{Symbol,Vector{Vector{Float64}}}()
    elap[:setup] = [elap_setup]
    elap[:rhs] = r_elap_rhs
    elap[:mem] = [[mem]]
    p_elap_main = PA.gather(map(_->elap,ranks))
    PA.map_main(p_elap_main) do p_elap
        JSON.json("$title.json",p_elap)
    end
    title
end


main (generic function with 1 method)

### Additional functions


In [44]:
function title(nx,ny,np)
   "results_nx$(nx)ny$(ny)np$np"
end

function main_mpi(nx,ny)
    PA.with_mpi() do backend
        comm = MPI.COMM_WORLD
        np = MPI.Comm_size(comm)
        main(backend,np,3,3,"warmup")
        MPI.Barrier(comm)
        main(backend,np,nx,ny,title(nx,ny,np))
    end
end

function main_debug(nx,ny,np)
    PA.with_debug() do backend
        main(backend,np,nx,ny,"debug")
    end
end

main_debug (generic function with 1 method)

### Testing
Calculate initial conditions and see parts of the matrix 

In [67]:
num_layer = 3

p_setup_on_main = PA.map_main(ranks) do _
    prepare_setups_on_main(3, 1, 1)
end

p_setup = PA.scatter(p_setup_on_main)

p_ln_u0_z0 = map(setup -> setup_ln_u0_z0(setup, num_layer), p_setup)

p_debug = PA.gather(p_ln_u0_z0)

PA.map_main(p_debug) do debug
    for rank in 1:length(debug)
        ln_u0, all_z_0 = debug[rank]

        println("rank = ", rank)
        println("ln_u0 = ", ln_u0)
        println("number of z layers = ", length(all_z_0))

        for layer in 1:length(all_z_0)
            println("layer = ", layer)
            println("size(z_0[layer]) = ", size(all_z_0[layer]))
            println("z_0[layer] = ")
            display(all_z_0[layer])
        end
    end
end;

25.0

24.238590728505365

24.238590728505365

23.514942928391992

24.805951036878145

...

24.423341501828236

23.683232500118713

24.423341501828236

23.683232500118713

24.23474244121035

In [11]:
gu_full = [2,2,2,2,2,2,2,2,2,2]
gn_ln_z1 = ones(10,10)
gn_ln_dz1 = copy(gn_ln_z1)


@elapsed for i in 1:10 # iterating column by column 
    for j in 1:10
        gn_ln_dz1[j,i] = gu_full[j] - gn_ln_z1[j,i] 
    end
end


3.04e-5

Initial conditions test 

In [46]:
np = 3
ngn = 4

backend = PA.DebugArray
ranks = backend(1:np)

p_ln_u0 = map(r -> [r], ranks)

gn_partition = PA.uniform_partition(ranks, np, ngn)
u0 = PA.PVector(p_ln_u0, gn_partition)

4-element PVector partitioned into 3 parts of type Vector{Int64}


Elap test

In [ ]:
r_elap_rhs = [zeros(7) for _ in 1:2]
print(r_elap_rhs)

z_0 = gn_0 * transpose(ln_u0)

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]

Z shape test

In [47]:
function φ(r;)
    α=5
    β=1/4
    α / (cosh(β*norm(r)))
end

function setup_ln_u0_z0(setup)
    (;ln_x, gn_x, num_layers) = setup 
    ln_u0 = φ.(ln_x) # initialize u
    gn_0 = φ.(gn_x)
    z0 = gn_0 * transpose(ln_u0) # initialize z layer
    ln_all_z0 = [copy(z0) for _ in 1:num_layers]
    return ln_u0, ln_all_z0      
end
ln_x = [(1,2), (3,4)]
gn_x = [(1,2), (3,4), (5,4), (6,5)]
num_layers = 3
setup = (; ln_x, gn_x, num_layers)
ln_u0, ln_all_z0 = setup_ln_u0_z0(setup)

([4.309001484146999, 2.6477106440302], [[18.567493790381043 11.408989094717938; 11.408989094717938 7.0103716545108155; 8.352977308093031 5.132573522972163; 5.994189199536241 3.6831917102680545], [18.567493790381043 11.408989094717938; 11.408989094717938 7.0103716545108155; 8.352977308093031 5.132573522972163; 5.994189199536241 3.6831917102680545], [18.567493790381043 11.408989094717938; 11.408989094717938 7.0103716545108155; 8.352977308093031 5.132573522972163; 5.994189199536241 3.6831917102680545]])

In [49]:
# print(ln_all_z0)
# length(ln_all_z0[1])
u0 = PA.PVector(p_ln_u0, gn_partition)
#z0 = PA.PMatrix(ln_all_z0, gn_partition)
length(u0)

4